# PVNet model smoke test: one 2020 sample

Run one real 2020 sample (GSP 199, 21 June 2020 at 12:00 UTC) through the repository's configured PVNet model. Inputs come from existing public satellite, UKV and PV_Live sampler stores. This checks batch compatibility and that the full forward path returns a forecast-shaped tensor.

The model has randomly initialized weights: this is an integration check, **not** a meaningful forecast or skill evaluation. It does not train, download data, write checkpoints, or change the repository configuration. Run cells from top to bottom.

## 1. Locate the existing sample stores

In [1]:
from pathlib import Path

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/model/late_fusion.yaml").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not locate the PVNet repository root from the notebook working directory.")

DIAGNOSTIC_DIR = REPO_ROOT / "Luke/data/to_gitignore/public_bucket_diagnostic"
SATELLITE_STORE = DIAGNOSTIC_DIR / "20200621T1200Z_GSP199_8frame_context_sampler.zarr"
UKV_STORE = DIAGNOSTIC_DIR / "ukv_20200621_06/ukv_gsp199_20200621_06_sampler.zarr"
PVLIVE_STORE = DIAGNOSTIC_DIR / "pvlive_gsp199_20200621T1200Z_context_sampler.zarr"
DATA_CONFIG_SOURCE = REPO_ROOT / "configs/datamodule/configuration/example_configuration.yaml"
MODEL_CONFIG_PATH = REPO_ROOT / "configs/model/late_fusion.yaml"

required = [SATELLITE_STORE, UKV_STORE, PVLIVE_STORE, DATA_CONFIG_SOURCE, MODEL_CONFIG_PATH]
missing = [p for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required input(s):\n" + "\n".join(map(str, missing)))

print("Repository:", REPO_ROOT)
print("Sample time: 2020-06-21 12:00 UTC; GSP 199")
for path in [SATELLITE_STORE, UKV_STORE, PVLIVE_STORE]:
    print("Found:", path)

Repository: /Users/mans5211/GitHub-repos/PVNet
Sample time: 2020-06-21 12:00 UTC; GSP 199
Found: /Users/mans5211/GitHub-repos/PVNet/Luke/data/to_gitignore/public_bucket_diagnostic/20200621T1200Z_GSP199_8frame_context_sampler.zarr
Found: /Users/mans5211/GitHub-repos/PVNet/Luke/data/to_gitignore/public_bucket_diagnostic/ukv_20200621_06/ukv_gsp199_20200621_06_sampler.zarr
Found: /Users/mans5211/GitHub-repos/PVNet/Luke/data/to_gitignore/public_bucket_diagnostic/pvlive_gsp199_20200621T1200Z_context_sampler.zarr


## 2. Point a temporary sampler configuration at the stores

This mirrors the windows used by the configured model: UKV from −2 to +2 hours, generation and solar context from −2 to +3 hours, and the existing satellite history. It writes a run-specific YAML beside the ignored diagnostic data; checked-in configs are untouched.

In [2]:
import yaml

with DATA_CONFIG_SOURCE.open() as f:
    data_config = yaml.safe_load(f)
input_data = data_config["input_data"]
input_data["nwp"] = {"ukv": input_data["nwp"]["ukv"]}

generation_cfg = input_data["generation"]
generation_cfg["zarr_path"] = str(PVLIVE_STORE.resolve())
generation_cfg["interval_start_minutes"] = -120
generation_cfg["interval_end_minutes"] = 180
generation_cfg["time_resolution_minutes"] = 30

ukv_cfg = input_data["nwp"]["ukv"]
ukv_cfg["zarr_path"] = str(UKV_STORE.resolve())
ukv_cfg["interval_start_minutes"] = -60
ukv_cfg["interval_end_minutes"] = 180
ukv_cfg["time_resolution_minutes"] = 60

input_data["satellite"]["zarr_path"] = str(SATELLITE_STORE.resolve())
solar_cfg = input_data["solar_position"]
solar_cfg["interval_start_minutes"] = -120
solar_cfg["interval_end_minutes"] = 180
solar_cfg["time_resolution_minutes"] = 30

RUN_CONFIG_PATH = DIAGNOSTIC_DIR / "pvnet_2020_model_smoke_configuration.yaml"
with RUN_CONFIG_PATH.open("w") as f:
    yaml.safe_dump(data_config, f, sort_keys=False)
print("Wrote temporary sampler config:", RUN_CONFIG_PATH)
print("Satellite channels:", len(input_data["satellite"]["channels"]))
print("UKV channels:", len(ukv_cfg["channels"]))

Wrote temporary sampler config: /Users/mans5211/GitHub-repos/PVNet/Luke/data/to_gitignore/public_bucket_diagnostic/pvnet_2020_model_smoke_configuration.yaml
Satellite channels: 11
UKV channels: 17


## 3. Assemble and inspect one real PVNetDataset sample

In [3]:
import pandas as pd
from ocf_data_sampler.torch_datasets.pvnet_dataset import PVNetDataset
from pvnet.datamodule import collate_fn

T0 = pd.Timestamp("2020-06-21 12:00:00")  # naive timestamps in these local stores are UTC
dataset = PVNetDataset(
    str(RUN_CONFIG_PATH),
    time_periods=[(T0.isoformat(sep=" "), T0.isoformat(sep=" "))],
)
if len(dataset) != 1:
    raise ValueError(f"Expected exactly one sample at {T0}; dataset contains {len(dataset)}.")
sample = dataset[0]
batch = collate_fn([sample])

print("Satellite:", tuple(batch["satellite_actual"].shape), " [batch, time, channels, y, x]")
print("UKV:", tuple(batch["nwp"]["ukv"]["nwp"].shape), " [batch, time, channels, y, x]")
print("Generation:", tuple(batch["generation"].shape))
print("Solar elevation:", tuple(batch["solar_elevation"].shape))
print("GSP ID:", batch["location_id"])

Satellite: (1, 7, 11, 24, 24)  [batch, time, channels, y, x]
UKV: (1, 5, 17, 24, 24)  [batch, time, channels, y, x]
Generation: (1, 11)
Solar elevation: (1, 11)
GSP ID: tensor([199])


## 4. Run the configured model in inference mode

The model config has an illustrative location-ID mapping. This one-GSP diagnostic maps GSP 199 to embedding index 0, as in the January smoke test. Evaluation mode allows batch size one for layers such as BatchNorm.

In [4]:
import torch
from hydra.utils import instantiate
from omegaconf import OmegaConf
from pvnet.utils import validate_batch_against_config

model_config = OmegaConf.load(MODEL_CONFIG_PATH)
model_config.model.location_id_mapping = {199: 0}
lightning_model = instantiate(model_config)
model = lightning_model.model.cpu().eval()
validate_batch_against_config(batch, model)

with torch.inference_mode():
    prediction = model(batch)

print("Input batch passed model-config validation.")
print("Prediction shape:", tuple(prediction.shape), " [batch, forecast steps, quantiles]")
print("Forecast steps:", model.forecast_len)
print("Quantiles:", model.output_quantiles)
print("All prediction values finite:", bool(torch.isfinite(prediction).all()))
print("Random-weight prediction range:", (float(prediction.min()), float(prediction.max())))

Input batch passed model-config validation.
Prediction shape: (1, 6, 3)  [batch, forecast steps, quantiles]
Forecast steps: 6
Quantiles: [0.1, 0.5, 0.9]
All prediction values finite: True
Random-weight prediction range: (-0.003362013027071953, 0.28300750255584717)


## What this establishes

If the final cell completes, the 2020 satellite, UKV and PV_Live sample can be assembled and passed through the configured PVNet model's forward path. This is a compatibility milestone before attempting a larger multi-day training run. The returned values are not realistic forecasts because the model starts from random weights. The next meaningful step is training on multiple dates and evaluating on held-out dates; forecast skill requires substantially more data and training.